In [ ]:
%pip install pandas plotly fpdf2 langgraph langchain google-generativeai


In [ ]:
dbutils.library.restartPython()


In [ ]:
"""
InsightForge AI — Supervisor Notebook
======================================
Orchestrates the full 6-agent pipeline using LangGraph.
Each agent has a single responsibility and communicates
through a shared typed state dictionary.

Author   : [Your Name]
Platform : Databricks Free Edition
Model    : gemini-flash-latest
Dataset  : /Volumes/insight/default/titanic/Titanic.csv
Version  : 1.0.0
"""

import time
import pandas as pd
import plotly.express as px
import google.generativeai as genai

from fpdf            import FPDF
from datetime        import datetime
from typing          import TypedDict, Any
from langgraph.graph import StateGraph, END

# ── Runtime configuration ─────────────────────────────────────
# These are set once here and used across all agents
# Never hardcode API keys — always use widgets

dbutils.widgets.text(
    "dataset_path",
    "/Volumes/insight/default/titanic/Titanic.csv",
    "Dataset Path"
)
dbutils.widgets.text(
    "gemini_key",
    "",
    "Gemini API Key"
)

DATASET_PATH = dbutils.widgets.get("dataset_path")
GEMINI_KEY   = dbutils.widgets.get("gemini_key")
GEMINI_MODEL = "gemini-flash-latest"

# ── Load dataset ──────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)

# ── Configure Gemini ──────────────────────────────────────────
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)

print("=" * 55)
print("  InsightForge AI — Supervisor")
print("=" * 55)
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Shape    : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Columns  : {df.columns.tolist()}")
print(f"  Model    : {GEMINI_MODEL}")
print(f"  API key  : {len(GEMINI_KEY)} characters")
print("=" * 55)


In [ ]:
"""
InsightForge AI — Logging Setup
================================
Configures structured logging for the full pipeline.
Every agent logs its start, completion, and any errors.

Log levels used:
- INFO    : normal operation messages
- WARNING : non-critical issues (rate limits, missing cols)
- ERROR   : agent failures that affect output quality
"""

import logging
import sys
from datetime import datetime

# ── Configure logger ──────────────────────────────────────────
def setup_logger(name: str = "InsightForge") -> logging.Logger:
    """
    Creates and configures a logger for the pipeline.
    Logs to console with timestamp, level, and message.
    Returns the same logger if already configured so
    calling setup_logger() multiple times is safe.

    Parameters
    ----------
    name : str — logger name, default InsightForge

    Returns
    -------
    logging.Logger : configured logger instance
    """
    logger = logging.getLogger(name)

    # Avoid adding duplicate handlers on re-run
    if logger.handlers:
        return logger

    logger.setLevel(logging.DEBUG)

    # ── Console handler ───────────────────────────────────────
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        fmt   = "%(asctime)s  [%(levelname)-8s]  %(message)s",
        datefmt = "%H:%M:%S"
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger


# ── Initialise logger ─────────────────────────────────────────
logger = setup_logger("InsightForge")

logger.info("InsightForge AI logging initialised")
logger.info(f"Dataset  : {DATASET_PATH}")
logger.info(f"Model    : {GEMINI_MODEL}")
logger.info(f"Log level: INFO")

print()
print("✅ Logger configured")
print("   Format: HH:MM:SS  [LEVEL]  message")


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def schema_agent(state: InsightForgeState) -> dict:
    """
    Agent 1 — Schema Agent
    ----------------------
    Responsibility: understand what the dataset is about.

    Reads  : state["raw_df"]
    Writes : state["schema_info"]
             state["pipeline_log"]

    Uses Gemini to detect the data domain, target variable,
    and industry from column names and sample values.
    Does NOT modify the DataFrame.
    """
    agent_name = "Schema Agent"
    print(f"\n{'─' * 55}")
    print(f"🔵 {agent_name} starting...")

    df     = state["raw_df"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    logger.info(
        f"{agent_name} started — "
        f"{df.shape[0]} rows x {df.shape[1]} cols"
    )

    try:
        # ── Structural analysis ───────────────────────────────
        numeric_cols     = list(df.select_dtypes(include="number").columns)
        categorical_cols = list(df.select_dtypes(include="object").columns)
        datetime_cols    = list(df.select_dtypes(include="datetime").columns)
        missing_by_col   = df.isnull().sum().to_dict()
        missing_pct      = (
            df.isnull().sum() / len(df) * 100
        ).round(2).to_dict()

        # ── Gemini domain detection ───────────────────────────
        prompt = f"""
You are a data scientist. Analyse these dataset details.

Column names : {list(df.columns)}
Data types   : {df.dtypes.astype(str).to_dict()}
Sample row   : {df.iloc[0].to_dict()}
Missing cols : {[c for c, v in missing_by_col.items() if v > 0]}

Reply in EXACTLY this format, one value per line:
DOMAIN: [type of data e.g. passenger records, sales transactions]
TARGET: [most likely target or outcome column name only]
INDUSTRY: [industry e.g. Transportation, Retail, Healthcare]
SUMMARY: [one sentence describing this dataset]
"""
        response = safe_call_gemini(prompt, agent_name)

        # ── Parse Gemini response ─────────────────────────────
        domain   = "Unknown"
        target   = ""
        industry = "Unknown"
        summary  = "No summary available"

        for line in response.split("\n"):
            line = line.strip()
            if line.startswith("DOMAIN:")  : domain   = line.replace("DOMAIN:",   "").strip()
            if line.startswith("TARGET:")  : target   = line.replace("TARGET:",   "").strip()
            if line.startswith("INDUSTRY:"): industry = line.replace("INDUSTRY:", "").strip()
            if line.startswith("SUMMARY:") : summary  = line.replace("SUMMARY:",  "").strip()

        # ── Build schema info dict ────────────────────────────
        schema_info = {
            "columns"         : list(df.columns),
            "dtypes"          : df.dtypes.astype(str).to_dict(),
            "numeric_cols"    : numeric_cols,
            "categorical_cols": categorical_cols,
            "datetime_cols"   : datetime_cols,
            "row_count"       : int(df.shape[0]),
            "col_count"       : int(df.shape[1]),
            "missing_by_col"  : missing_by_col,
            "missing_pct"     : missing_pct,
            "domain"          : domain,
            "target_variable" : target,
            "industry"        : industry,
            "summary"         : summary,
        }

        log = log_event(
            state, agent_name,
            f"done — domain={domain}, target={target}"
        )
        logger.info(
            f"{agent_name} complete — "
            f"domain={domain}, target={target}"
        )
        print(f"   Domain   : {domain}")
        print(f"   Target   : {target}")
        print(f"   Industry : {industry}")
        print(f"   Summary  : {summary[:60]}...")
        print(f"✅ {agent_name} complete")

        return {
            "schema_info"  : schema_info,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        logger.error(f"{agent_name} FAILED — {str(e)}")
        msg = f"{agent_name} failed: {str(e)}"
        print(f"   ❌ {msg}")
        return {
            "schema_info"  : {},
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ schema_agent() defined")


In [ ]:
def cleaning_agent(state: InsightForgeState) -> dict:
    """
    Agent 2 — Cleaning Agent
    ------------------------
    Reads  : state["raw_df"]
    Writes : state["cleaned_df"], state["cleaning_report"]
    """
    agent_name = "Cleaning Agent"
    print(f"\n{'─' * 55}")
    print(f"🟡 {agent_name} starting...")

    df     = state["raw_df"].copy()
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    logger.info(
        f"{agent_name} started — "
        f"{df.shape[0]} rows, {df.shape[1]} cols"
    )

    report = {
        "nulls_filled"       : {},
        "columns_dropped"    : [],
        "duplicates_removed" : 0,
        "outliers_flagged"   : {},
        "rows_before"        : int(len(df)),
        "rows_after"         : 0,
        "cols_before"        : int(df.shape[1]),
        "cols_after"         : 0,
    }

    try:
        for col in df.select_dtypes(include="number").columns:
            null_count = int(df[col].isnull().sum())
            if null_count > 0:
                fill_value = df[col].median()
                df[col]    = df[col].fillna(fill_value)
                report["nulls_filled"][col] = {
                    "strategy" : "median",
                    "value"    : round(float(fill_value), 4),
                    "count"    : null_count
                }
                print(
                    f"   Filled {null_count:4} nulls in "
                    f"'{col}' → median = {round(fill_value, 2)}"
                )

        for col in df.select_dtypes(include="object").columns:
            null_count  = int(df[col].isnull().sum())
            missing_pct = null_count / len(df) * 100
            if missing_pct > 70:
                df = df.drop(columns=[col])
                report["columns_dropped"].append(col)
                print(f"   Dropped '{col}' — {round(missing_pct, 1)}% missing")
                logger.warning(
                    f"{agent_name} — dropped '{col}' "
                    f"({round(missing_pct,1)}% missing)"
                )
            elif null_count > 0:
                fill_value = df[col].mode()[0]
                df[col]    = df[col].fillna(fill_value)
                report["nulls_filled"][col] = {
                    "strategy" : "mode",
                    "value"    : str(fill_value),
                    "count"    : null_count
                }
                print(
                    f"   Filled {null_count:4} nulls in "
                    f"'{col}' → mode = '{fill_value}'"
                )

        rows_before   = len(df)
        df            = df.drop_duplicates()
        dupes_removed = rows_before - len(df)
        report["duplicates_removed"] = dupes_removed

        if dupes_removed > 0:
            print(f"   Removed {dupes_removed} duplicate rows")
            logger.warning(
                f"{agent_name} — removed {dupes_removed} duplicates"
            )
        else:
            print(f"   No duplicate rows found")

        for col in df.select_dtypes(include="number").columns:
            q1  = df[col].quantile(0.25)
            q3  = df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_count = int(
                ((df[col] < lower) | (df[col] > upper)).sum()
            )
            if outlier_count > 0:
                report["outliers_flagged"][col] = {
                    "count"       : outlier_count,
                    "lower_bound" : round(float(lower), 4),
                    "upper_bound" : round(float(upper), 4)
                }

        report["rows_after"] = int(len(df))
        report["cols_after"] = int(df.shape[1])

        log = log_event(
            state, agent_name,
            f"done — {len(report['nulls_filled'])} cols filled, "
            f"{len(report['columns_dropped'])} dropped, "
            f"{dupes_removed} dupes removed"
        )

        logger.info(
            f"{agent_name} complete — "
            f"{len(report['nulls_filled'])} cols filled, "
            f"{len(report['columns_dropped'])} dropped, "
            f"{dupes_removed} dupes removed"
        )

        print(f"   Rows : {report['rows_before']} → {report['rows_after']}")
        print(f"   Cols : {report['cols_before']} → {report['cols_after']}")
        print(f"✅ {agent_name} complete")

        return {
            "cleaned_df"     : df,
            "cleaning_report": report,
            "pipeline_log"   : log,
            "errors"         : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "cleaned_df"     : state["raw_df"],
            "cleaning_report": report,
            "pipeline_log"   : log_event(
                state, agent_name, f"FAILED — {e}"
            ),
            "errors"         : errors + [msg]
        }

print("✅ cleaning_agent() defined")


In [ ]:
def eda_agent(state: InsightForgeState) -> dict:
    """
    Agent 3 — EDA Agent
    -------------------
    Reads  : state["cleaned_df"], state["schema_info"]
    Writes : state["eda_results"]
    """
    agent_name = "EDA Agent"
    print(f"\n{'─' * 55}")
    print(f"🟠 {agent_name} starting...")

    df     = state["cleaned_df"]
    schema = state["schema_info"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")
    target = schema.get("target_variable", "")

    logger.info(f"{agent_name} started — {df.shape}")

    try:
        numeric_cols = list(df.select_dtypes(include="number").columns)
        cat_cols     = list(df.select_dtypes(include="object").columns)

        describe = df.describe().round(4).to_dict()
        print(f"   Descriptive stats : {len(numeric_cols)} numeric columns")

        if len(numeric_cols) >= 2:
            corr_matrix = df[numeric_cols].corr().round(4).to_dict()
            print(f"   Correlation matrix: {len(numeric_cols)}x{len(numeric_cols)}")
        else:
            corr_matrix = {}

        top_correlations = {}
        if target and target in numeric_cols and len(numeric_cols) >= 2:
            top_correlations = (
                df[numeric_cols].corr()[target]
                .drop(target)
                .abs()
                .sort_values(ascending=False)
                .round(4)
                .to_dict()
            )
            print(f"   Top correlations  : computed for '{target}'")

        value_counts = {
            col: df[col].value_counts().head(10).to_dict()
            for col in cat_cols
        }
        print(f"   Value counts      : {len(cat_cols)} categorical columns")

        skewness = {
            col: round(float(df[col].skew()), 4)
            for col in numeric_cols
        }
        kurtosis = {
            col: round(float(df[col].kurt()), 4)
            for col in numeric_cols
        }

        total_nulls = int(df.isnull().sum().sum())

        eda_results = {
            "shape"            : list(df.shape),
            "numeric_cols"     : numeric_cols,
            "categorical_cols" : cat_cols,
            "describe"         : describe,
            "correlation"      : corr_matrix,
            "top_correlations" : top_correlations,
            "value_counts"     : value_counts,
            "skewness"         : skewness,
            "kurtosis"         : kurtosis,
            "remaining_nulls"  : df.isnull().sum().to_dict(),
            "total_nulls"      : total_nulls,
        }

        log = log_event(
            state, agent_name,
            f"done — shape {df.shape}, {total_nulls} nulls remaining"
        )

        logger.info(
            f"{agent_name} complete — "
            f"{len(numeric_cols)} numeric, "
            f"{len(cat_cols)} categorical, "
            f"{total_nulls} nulls remaining"
        )

        print(f"✅ {agent_name} complete")

        return {
            "eda_results"  : eda_results,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "eda_results"  : {},
            "pipeline_log" : log_event(
                state, agent_name, f"FAILED — {e}"
            ),
            "errors"       : errors + [msg]
        }

print("✅ eda_agent() defined")


In [ ]:
def viz_agent(state: InsightForgeState) -> dict:
    """
    Agent 4 — Visualization Agent
    ------------------------------
    Reads  : state["cleaned_df"], state["schema_info"],
             state["eda_results"]
    Writes : state["charts"]
    """
    agent_name = "Visualization Agent"
    print(f"\n{'─' * 55}")
    print(f"🟣 {agent_name} starting...")

    df     = state["cleaned_df"]
    schema = state["schema_info"]
    eda    = state["eda_results"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    target       = schema.get("target_variable", "")
    numeric_cols = [
        c for c in eda.get("numeric_cols", []) if c in df.columns
    ]
    cat_cols = [
        c for c in eda.get("categorical_cols", []) if c in df.columns
    ]

    logger.info(
        f"{agent_name} started — "
        f"{len(numeric_cols)} numeric, {len(cat_cols)} categorical"
    )

    charts = []

    try:
        for col in numeric_cols[:4]:
            fig = px.histogram(
                df, x=col,
                title                   = f"Distribution - {col}",
                template                = "plotly_white",
                color_discrete_sequence = ["#4C72B0"]
            )
            fig.update_layout(xaxis_title=col, yaxis_title="Count")
            charts.append({"type":"histogram","title":f"Distribution - {col}","col":col,"fig":fig})
            fig.show()
            print(f"   Chart: histogram — {col}")

        for col in cat_cols[:3]:
            vc = df[col].value_counts().head(10).reset_index()
            vc.columns = [col, "Count"]
            fig = px.bar(
                vc, x=col, y="Count",
                title    = f"Value Counts - {col}",
                template = "plotly_white",
                color    = col, text="Count"
            )
            fig.update_traces(textposition="outside")
            fig.update_layout(showlegend=False)
            charts.append({"type":"bar","title":f"Value Counts - {col}","col":col,"fig":fig})
            fig.show()
            print(f"   Chart: bar — {col}")

        if len(numeric_cols) >= 2:
            corr = df[numeric_cols].corr().round(2)
            fig  = px.imshow(
                corr, text_auto=True,
                title                  = "Correlation Heatmap",
                color_continuous_scale = "RdBu_r",
                template               = "plotly_white",
                zmin=-1, zmax=1
            )
            fig.update_layout(width=650, height=500)
            charts.append({"type":"heatmap","title":"Correlation Heatmap","col":"all","fig":fig})
            fig.show()
            print(f"   Chart: correlation heatmap")

        if target and target in df.columns:
            for col in cat_cols[:2]:
                group = (
                    df.groupby(col)[target]
                    .mean()
                    .reset_index()
                    .rename(columns={target: f"{target} Rate"})
                )
                group[f"{target} Rate"] = group[f"{target} Rate"].round(4)
                fig = px.bar(
                    group, x=col, y=f"{target} Rate",
                    title    = f"{target} Rate by {col}",
                    template = "plotly_white",
                    color=col, text=f"{target} Rate"
                )
                fig.update_traces(textposition="outside")
                fig.update_layout(yaxis_range=[0,1.15], showlegend=False)
                charts.append({"type":"bar","title":f"{target} Rate by {col}","col":col,"fig":fig})
                fig.show()
                print(f"   Chart: {target} rate by {col}")

        if "Age" in df.columns and "Fare" in df.columns:
            color_col = target if target in df.columns else None
            fig = px.scatter(
                df, x="Age", y="Fare", color=color_col,
                title    = f"Age vs Fare{' - coloured by '+target if color_col else ''}",
                template = "plotly_white", opacity=0.65
            )
            fig.update_traces(marker=dict(size=5))
            charts.append({"type":"scatter","title":"Age vs Fare","col":"Age_Fare","fig":fig})
            fig.show()
            print(f"   Chart: scatter — Age vs Fare")

        log = log_event(
            state, agent_name,
            f"done — {len(charts)} charts generated"
        )

        logger.info(f"{agent_name} complete — {len(charts)} charts generated")

        print(f"   Total : {len(charts)} charts")
        print(f"✅ {agent_name} complete")

        return {
            "charts"       : charts,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "charts"       : [],
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ viz_agent() defined")


In [ ]:
def insight_agent(state: InsightForgeState) -> dict:
    """
    Agent 5 — Insight Agent
    -----------------------
    Reads  : state["cleaned_df"], state["schema_info"],
             state["eda_results"], state["cleaning_report"]
    Writes : state["insights"]
    """
    agent_name = "Insight Agent"
    print(f"\n{'─' * 55}")
    print(f"🔴 {agent_name} starting...")

    df     = state["cleaned_df"]
    schema = state["schema_info"]
    eda    = state["eda_results"]
    clean  = state["cleaning_report"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")
    target = schema.get("target_variable", "")

    logger.info(
        f"{agent_name} started — "
        f"sending to {GEMINI_MODEL}"
    )

    try:
        corr_lines = []
        if target and target in eda.get("correlation", {}):
            target_corr = {
                k: v for k, v in eda["correlation"][target].items()
                if k != target
            }
            sorted_corr = sorted(
                target_corr.items(),
                key=lambda x: abs(x[1]), reverse=True
            )
            corr_lines = [
                f"  {col}: {val:+.4f}"
                for col, val in sorted_corr[:6]
            ]

        vc_lines = []
        for col, counts in eda.get("value_counts", {}).items():
            vc_lines.append(f"\n  {col}:")
            for val, count in list(counts.items())[:5]:
                pct = round(count / eda["shape"][0] * 100, 1)
                vc_lines.append(f"    {str(val)}: {count} ({pct}%)")

        filled_summary = ", ".join([
            f"{col} ({info['count']} via {info['strategy']})"
            for col, info in clean.get("nulls_filled", {}).items()
        ])

        prompt = f"""
You are a senior data scientist and business analyst.
Analyse this dataset and provide a professional report.

DATASET CONTEXT
Domain   : {schema.get("domain",   "Unknown")}
Industry : {schema.get("industry", "Unknown")}
Summary  : {schema.get("summary",  "Unknown")}
Shape    : {eda["shape"][0]} rows x {eda["shape"][1]} columns
Target   : {target if target else "Not specified"}

DATA QUALITY ACTIONS TAKEN
Rows before cleaning   : {clean.get("rows_before", "N/A")}
Rows after cleaning    : {clean.get("rows_after",  "N/A")}
Duplicate rows removed : {clean.get("duplicates_removed", 0)}
Columns dropped        : {clean.get("columns_dropped", [])}
Null values filled     : {filled_summary or "None"}

STATISTICAL SUMMARY
{df.describe().round(2).to_string()}

CORRELATIONS WITH TARGET ({target.upper()})
{chr(10).join(corr_lines) if corr_lines else "No target variable specified"}

VALUE DISTRIBUTIONS
{chr(10).join(vc_lines)}

SKEWNESS
{chr(10).join([f"  {k}: {v}" for k, v in eda.get("skewness", {}).items()])}

SAMPLE DATA (first 5 rows)
{df.head(5).to_string()}

YOUR TASK
Write a professional analysis with EXACTLY these 6 sections.
Use actual numbers. No generic statements.

1. EXECUTIVE SUMMARY (3-4 sentences)
2. KEY FINDINGS (exactly 5 bullet points with numbers)
3. BUSINESS INSIGHTS (3-4 actionable recommendations)
4. DATA QUALITY ASSESSMENT
5. PATTERNS AND ANOMALIES
6. RECOMMENDED NEXT STEPS (3 specific actions)
"""

        print(f"   Prompt length  : {len(prompt):,} characters")
        print(f"   Sending to {GEMINI_MODEL}...")

        insights = safe_call_gemini(prompt, agent_name)

        log = log_event(
            state, agent_name,
            f"done — {len(insights):,} characters returned"
        )

        logger.info(
            f"{agent_name} complete — "
            f"{len(insights):,} characters returned"
        )

        print(f"   Response : {len(insights):,} characters")
        print(f"✅ {agent_name} complete")

        return {
            "insights"     : insights,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "insights"     : "",
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ insight_agent() defined")


In [ ]:
def report_agent(state: InsightForgeState) -> dict:
    """
    Agent 6 — Report Agent
    ----------------------
    Reads  : all state fields
    Writes : state["pdf_path"]

    PDF sections:
    1. Dataset Overview
    2. Cleaning Report
    3. Statistical Summary table
    4. Key Correlations
    5. AI Insights
    6. Categorical Summary
    7. Pipeline Execution Log
    """
    agent_name = "Report Agent"
    print(f"\n{'─' * 55}")
    print(f"📄 {agent_name} starting...")

    df       = state["cleaned_df"]
    schema   = state["schema_info"]
    eda      = state["eda_results"]
    clean    = state["cleaning_report"]
    insights = state["insights"]
    pipe_log = state["pipeline_log"]
    errors   = state["errors"]
    log      = log_event(state, agent_name, "started")

    logger.info(f"{agent_name} started — assembling PDF")

    def clean_text(text) -> str:
        """Sanitises string for Helvetica font in fpdf2."""
        if not isinstance(text, str):
            text = str(text)
        replacements = {
            "\u2014":"-", "\u2013":"-", "\u2012":"-",
            "\u2011":"-", "\u2010":"-", "\u2022":"-",
            "\u2023":"-", "\u2043":"-", "\u2018":"'",
            "\u2019":"'", "\u201a":"'", "\u201c":'"',
            "\u201d":'"', "\u201e":'"', "\u2026":"...",
            "\u00a0":" ", "\u00b7":"-", "\u2015":"-",
            "\u2192":"->","\u2190":"<-","\u00d7":"x",
            "\u00f7":"/",
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    def safe_cell_value(raw_val) -> str:
        """Converts any value to a PDF-safe string."""
        try:
            if pd.isna(raw_val):
                return ""
        except (TypeError, ValueError):
            pass
        if raw_val in (float("inf"), float("-inf")):
            return "inf"
        if isinstance(raw_val, float):
            return clean_text(str(round(raw_val, 2)))
        return clean_text(str(raw_val))

    try:
        class InsightReport(FPDF):
            def header(self):
                self.set_fill_color(31, 97, 141)
                self.set_text_color(255, 255, 255)
                self.set_font("Helvetica", "B", 15)
                self.cell(
                    0, 13,
                    "InsightForge AI - Data Analysis Report",
                    align="C", fill=True,
                    new_x="LMARGIN", new_y="NEXT"
                )
                self.set_font("Helvetica", "", 8)
                self.set_text_color(120, 120, 120)
                self.cell(
                    0, 5,
                    f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}"
                    f"  |  {GEMINI_MODEL}"
                    f"  |  Databricks  |  LangGraph",
                    align="C",
                    new_x="LMARGIN", new_y="NEXT"
                )
                self.set_text_color(0, 0, 0)
                self.ln(3)

            def footer(self):
                self.set_y(-14)
                self.set_font("Helvetica", "I", 8)
                self.set_text_color(150, 150, 150)
                self.cell(
                    0, 10,
                    clean_text(
                        f"InsightForge AI  |  Page {self.page_no()}"
                        f"  |  LangGraph + Gemini on Databricks"
                    ),
                    align="C"
                )

            def section_header(self, title: str):
                self.set_font("Helvetica", "B", 11)
                self.set_fill_color(213, 232, 252)
                self.set_text_color(25, 80, 140)
                self.cell(
                    0, 8, clean_text(f"  {title}"),
                    fill=True, new_x="LMARGIN", new_y="NEXT"
                )
                self.set_text_color(0, 0, 0)
                self.ln(2)

            def key_value_row(self, label: str, value):
                self.set_font("Helvetica", "B", 10)
                self.cell(65, 7, clean_text(f"  {label}"), border="B")
                self.set_font("Helvetica", "", 10)
                self.cell(
                    0, 7,
                    safe_cell_value(value)[:100],
                    border="B", new_x="LMARGIN", new_y="NEXT"
                )

        pdf = InsightReport()
        pdf.add_page()

        # Section 1 — Dataset Overview
        pdf.section_header("1.  Dataset Overview")
        pdf.key_value_row("Domain",          schema.get("domain",  ""))
        pdf.key_value_row("Industry",         schema.get("industry",""))
        pdf.key_value_row("Summary",          schema.get("summary", ""))
        pdf.key_value_row("Rows",             eda["shape"][0])
        pdf.key_value_row("Columns",          eda["shape"][1])
        pdf.key_value_row("Target Variable",  schema.get("target_variable",""))
        pdf.key_value_row("Numeric Columns",  ", ".join(eda.get("numeric_cols",[])))
        pdf.key_value_row("Categorical Cols", ", ".join(eda.get("categorical_cols",[])))
        pdf.key_value_row("Remaining Nulls",  eda.get("total_nulls", 0))
        pdf.ln(4)

        # Section 2 — Cleaning Report
        pdf.section_header("2.  Data Cleaning Report")
        pdf.key_value_row("Rows Before",       clean.get("rows_before",""))
        pdf.key_value_row("Rows After",        clean.get("rows_after",""))
        pdf.key_value_row("Duplicates Removed",clean.get("duplicates_removed",0))
        pdf.key_value_row(
            "Columns Dropped",
            ", ".join(clean.get("columns_dropped",[])) or "None"
        )
        pdf.ln(2)
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, "  Null Filling Actions:", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 9)
        for col, info in clean.get("nulls_filled",{}).items():
            pdf.cell(
                0, 5,
                clean_text(
                    f"    {col}: {info['count']} nulls filled"
                    f" using {info['strategy']} = {info['value']}"
                ),
                new_x="LMARGIN", new_y="NEXT"
            )
        pdf.ln(2)
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, "  Outliers Flagged (IQR):", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 9)
        for col, info in clean.get("outliers_flagged",{}).items():
            pdf.cell(
                0, 5,
                clean_text(
                    f"    {col}: {info['count']} rows outside"
                    f" [{info['lower_bound']}, {info['upper_bound']}]"
                ),
                new_x="LMARGIN", new_y="NEXT"
            )
        pdf.ln(4)

        # Section 3 — Stats table
        pdf.section_header("3.  Statistical Summary")
        stats_df = df.describe().round(2).fillna("")
        n_cols   = len(stats_df.columns)
        col_w    = min(24, 165 / (n_cols + 1))

        pdf.set_font("Helvetica", "B", 8)
        pdf.set_fill_color(31, 97, 141)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(col_w, 6, "Stat", border=1, fill=True)
        for col in stats_df.columns:
            pdf.cell(col_w, 6, clean_text(str(col)[:9]), border=1, fill=True)
        pdf.ln()

        pdf.set_text_color(0, 0, 0)
        for i, idx in enumerate(stats_df.index):
            fill_color = (240, 247, 255) if i % 2 == 0 else (255, 255, 255)
            pdf.set_fill_color(*fill_color)
            pdf.set_font("Helvetica", "B", 8)
            pdf.cell(col_w, 5, clean_text(str(idx)), border=1, fill=True)
            pdf.set_font("Helvetica", "", 8)
            for col in stats_df.columns:
                pdf.cell(
                    col_w, 5,
                    safe_cell_value(stats_df.loc[idx, col]),
                    border=1, fill=True
                )
            pdf.ln()
        pdf.ln(4)

        # Section 4 — Correlations
        pdf.section_header("4.  Key Correlations")
        target = schema.get("target_variable","")
        corr   = eda.get("correlation",{})
        pdf.set_font("Helvetica", "", 10)
        if target and target in corr:
            target_corr = {
                k: v for k, v in corr[target].items() if k != target
            }
            sorted_corr = sorted(
                target_corr.items(),
                key=lambda x: abs(x[1]), reverse=True
            )
            for col, val in sorted_corr[:7]:
                try:
                    if pd.isna(val): continue
                except (TypeError, ValueError):
                    pass
                direction = "positive" if val > 0 else "negative"
                strength  = (
                    "strong"   if abs(val) > 0.4 else
                    "moderate" if abs(val) > 0.2 else "weak"
                )
                bar = "I" * int(abs(val) * 15)
                pdf.cell(
                    0, 6,
                    clean_text(
                        f"  {col:18} {val:+.4f}"
                        f"  ({strength} {direction})  {bar}"
                    ),
                    new_x="LMARGIN", new_y="NEXT"
                )
        pdf.ln(4)

        # Section 5 — AI Insights
        pdf.add_page()
        pdf.section_header(f"5.  AI Insights ({GEMINI_MODEL})")
        pdf.set_font("Helvetica", "", 10)
        pdf.multi_cell(0, 5, clean_text(insights))
        pdf.ln(4)

        # Section 6 — Categorical Summary
        pdf.section_header("6.  Categorical Column Summary")
        for col, counts in eda.get("value_counts",{}).items():
            pdf.set_font("Helvetica", "B", 10)
            pdf.cell(0, 7, clean_text(f"  {col}"), new_x="LMARGIN", new_y="NEXT")
            pdf.set_font("Helvetica", "", 9)
            for val, count in list(counts.items())[:6]:
                pct = round(count / eda["shape"][0] * 100, 1)
                bar = "I" * int(pct / 5)
                pdf.cell(
                    0, 5,
                    clean_text(f"    {str(val):18} : {count:5}  ({pct}%)  {bar}"),
                    new_x="LMARGIN", new_y="NEXT"
                )
            pdf.ln(2)

        # Section 7 — Pipeline Log
        pdf.add_page()
        pdf.section_header("7.  Pipeline Execution Log")
        pdf.set_font("Helvetica", "", 9)
        for entry in pipe_log:
            pdf.cell(
                0, 5, clean_text(f"  {entry}"),
                new_x="LMARGIN", new_y="NEXT"
            )

        if state["errors"]:
            pdf.ln(4)
            pdf.section_header("  Errors Encountered")
            pdf.set_font("Helvetica", "", 9)
            for err in state["errors"]:
                pdf.cell(
                    0, 5, clean_text(f"  ERROR: {err}"),
                    new_x="LMARGIN", new_y="NEXT"
                )

        # Save PDF — write directly to Volume (no /tmp on serverless)
        filename    = "insightforge_report.pdf"
        volume_path = f"/Volumes/insight/default/titanic/{filename}"

        pdf.output(volume_path)

        log = log_event(
            state, agent_name,
            f"done - PDF saved, {pdf.page_no()} pages"
        )

        logger.info(
            f"{agent_name} complete — "
            f"PDF saved to {volume_path}, "
            f"{pdf.page_no()} pages"
        )

        print(f"   Pages    : {pdf.page_no()}")
        print(f"   Saved to : {volume_path}")
        print(f"✅ {agent_name} complete")

        return {
            "pdf_path"     : volume_path,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "pdf_path"     : "",
            "pipeline_log" : log_event(
                state, agent_name, f"FAILED - {e}"
            ),
            "errors"       : errors + [msg]
        }

print("✅ report_agent() defined")


In [ ]:
def build_supervisor():
    """
    Builds and compiles the InsightForge AI LangGraph pipeline.

    Graph order:
    schema -> cleaning -> eda -> viz -> insight -> report -> END
    """
    graph = StateGraph(InsightForgeState)

    graph.add_node("schema",   schema_agent)
    graph.add_node("cleaning", cleaning_agent)
    graph.add_node("eda",      eda_agent)
    graph.add_node("viz",      viz_agent)
    graph.add_node("insight",  insight_agent)
    graph.add_node("report",   report_agent)

    graph.set_entry_point("schema")
    graph.add_edge("schema",   "cleaning")
    graph.add_edge("cleaning", "eda")
    graph.add_edge("eda",      "viz")
    graph.add_edge("viz",      "insight")
    graph.add_edge("insight",  "report")
    graph.add_edge("report",    END)

    return graph.compile()


supervisor = build_supervisor()

logger.info("Supervisor graph compiled — 6 agents registered")
print("✅ Supervisor compiled")
print("   schema -> cleaning -> eda -> viz -> insight -> report -> END")


In [ ]:
def run_pipeline(df: pd.DataFrame):
    """
    Runs the full InsightForge AI pipeline on a given DataFrame.

    Parameters
    ----------
    df : pd.DataFrame — the raw dataset to analyse

    Returns
    -------
    tuple : (final_state, elapsed_seconds)
    """
    initial_state = InsightForgeState(
        dataset_path    = DATASET_PATH,
        gemini_key      = GEMINI_KEY,
        raw_df          = df,
        cleaned_df      = None,
        schema_info     = {},
        cleaning_report = {},
        eda_results     = {},
        charts          = [],
        insights        = "",
        pdf_path        = "",
        pipeline_log    = [],
        errors          = []
    )

    logger.info("=" * 40)
    logger.info("InsightForge AI Pipeline Starting")
    logger.info(f"Dataset : {DATASET_PATH}")
    logger.info(f"Shape   : {df.shape[0]} rows x {df.shape[1]} cols")
    logger.info(f"Model   : {GEMINI_MODEL}")
    logger.info("=" * 40)

    print("=" * 55)
    print("  InsightForge AI Pipeline Starting")
    print("=" * 55)
    print(f"  Dataset : {DATASET_PATH}")
    print(f"  Shape   : {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  Model   : {GEMINI_MODEL}")
    print("=" * 55)

    start_time = time.time()
    final      = supervisor.invoke(initial_state)
    elapsed    = round(time.time() - start_time, 2)

    logger.info("Pipeline execution complete")
    logger.info(f"Total time : {elapsed} seconds")
    logger.info(f"Errors     : {len(final['errors'])}")

    if final["errors"]:
        for err in final["errors"]:
            logger.error(f"Pipeline error: {err}")
    else:
        logger.info("All 6 agents completed with 0 errors")

    return final, elapsed


# ── Execute ───────────────────────────────────────────────────
final, elapsed = run_pipeline(df)


In [ ]:
print()
print("=" * 55)
print("  INSIGHTFORGE AI — PIPELINE COMPLETE")
print("=" * 55)
print()

agents_summary = [
    (
        "Schema Agent",
        bool(final.get("schema_info")),
        f"domain = {str(final.get('schema_info',{}).get('domain',''))[:40]}"
    ),
    (
        "Cleaning Agent",
        bool(final.get("cleaning_report")),
        f"{len(final.get('cleaning_report',{}).get('nulls_filled',{}))} cols fixed, "
        f"{final.get('cleaning_report',{}).get('duplicates_removed',0)} dupes removed"
    ),
    (
        "EDA Agent",
        bool(final.get("eda_results")),
        f"shape = {final.get('eda_results',{}).get('shape','')}"
    ),
    (
        "Visualization Agent",
        bool(final.get("charts")),
        f"{len(final.get('charts',[]))} charts generated"
    ),
    (
        "Insight Agent",
        bool(final.get("insights")),
        f"{len(final.get('insights','')):,} characters"
    ),
    (
        "Report Agent",
        bool(final.get("pdf_path")),
        f"saved to {final.get('pdf_path','')}"
    ),
]

for name, success, detail in agents_summary:
    icon = "✅" if success else "❌"
    print(f"  {icon}  {name:22} — {detail}")

print()
print(f"  Total time : {elapsed} seconds")
print(f"  Errors     : {len(final.get('errors',[]))}")

if final.get("errors"):
    print()
    print("  Error details:")
    for err in final.get("errors",[]):
        print(f"    • {err}")

print()
print("  Pipeline execution log:")
for entry in final.get("pipeline_log",[]):
    print(f"    {entry}")

print()
print("=" * 55)
print(f"  PDF saved to : {final.get('pdf_path','')}")
print("=" * 55)


In [ ]:
from IPython.display import HTML

volume_path = final["pdf_path"]
folder_path = "/".join(volume_path.split("/")[:-1])
filename    = volume_path.split("/")[-1]

print("Checking for PDF report...")

try:
    files   = dbutils.fs.ls(folder_path)
    matched = [f for f in files if filename in f.name]

    if matched:
        size_kb = round(matched[0].size / 1024, 2)
        print(f"  ✅ {filename}  —  {size_kb} KB")
        print(f"  📍 {volume_path}")

        display(HTML(f"""
        <div style="padding:20px;
                    background:linear-gradient(135deg,#1a6b9a,#2196F3);
                    border-radius:10px;color:white;margin:15px 0;
                    font-family:Arial,sans-serif">
            <h3 style="margin:0 0 15px 0">
                InsightForge AI - Report Generated
            </h3>
            <div style="background:rgba(255,255,255,0.15);
                        padding:12px;border-radius:6px;margin:8px 0">
                <strong>Location:</strong><br/>
                <code style="color:#fff;background:rgba(0,0,0,0.25);
                             padding:4px 8px;border-radius:3px;
                             display:inline-block;margin-top:4px">
                    {volume_path}
                </code>
            </div>
            <div style="background:rgba(255,255,255,0.15);
                        padding:12px;border-radius:6px;margin:8px 0">
                <strong>File size:</strong> {size_kb} KB
            </div>
            <div style="background:rgba(255,255,255,0.95);
                        padding:15px;border-radius:6px;
                        color:#333;margin-top:12px">
                <strong style="color:#1a6b9a">How to download:</strong>
                <ol style="margin:8px 0 0 0;
                           padding-left:20px;line-height:1.9">
                    <li>Click <strong>Catalog</strong> in left sidebar</li>
                    <li>Navigate to
                        <strong>insight &rarr; default &rarr; titanic</strong>
                    </li>
                    <li>Find <strong>{filename}</strong></li>
                    <li>Click <strong>Download</strong></li>
                </ol>
            </div>
        </div>
        """))
    else:
        print(f"  ❌ {filename} not found in {folder_path}")

except Exception as e:
    print(f"  ❌ Error: {e}")
    print(f"     Folder: {folder_path}")
    


In [ ]:
# Add this guard so it only runs when you explicitly want it
CREATE_TABLE = False   # change to True only when needed

if CREATE_TABLE:
    spark.sql("CREATE SCHEMA IF NOT EXISTS insightforge")
    df_fresh = pd.read_csv("/Volumes/insight/default/titanic/Titanic.csv")
    spark_df  = spark.createDataFrame(df_fresh)
    spark_df.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable("insightforge.titanic_cleaned")
    print("✅ Delta table created")
else:
    print("⏭️  Skipping table creation — set CREATE_TABLE=True to run")


In [ ]:
# See all catalogs
display(spark.sql("SHOW CATALOGS"))


In [ ]:
# See all schemas in your catalog
display(spark.sql("SHOW SCHEMAS IN insight"))
